# Unsupervised exploration (_Iris_)

In [1]:
from experiments.utils.constants import RANDOM_SEED, SOM_LEARNING_RATE_DECAY_FN, SOM_FIT_METHOD

VERBOSE = True
DATASET_NAME = "Iris"
DATASET_ID = 53
MODELING_MODE = False
MODEL_REGISTRATION_MODE = False

## Dataset

In [2]:
# fetch dataset
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=DATASET_ID)
X = dataset.data.features.values
y = dataset.data.targets.values.ravel()
print(f"Dataset shape: {X.shape}, {y.shape}")

Dataset shape: (150, 4), (150,)


In [3]:
# scale data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
del X

## Modeling

In [4]:
from minisom_representation import calc_som_hyparams, SomRepresentation, plot_som_convergence_over_epochs

In [5]:
# use helper methods to get SOM hyperparameter recommendations
recommended_params = calc_som_hyparams(X_scaled, initial_sigma_factor=3.0)
print("Recommended SOM parameters:", recommended_params)

Recommended SOM parameters: {'d1': 6, 'd2': 11, 'sigma': 3.67}


In [6]:
# define actual hyperparameters
d1, d2, sigma = map(recommended_params.get, ("d1", "d2", "sigma"))
decay_function = SOM_LEARNING_RATE_DECAY_FN
epoch = None

In [7]:
# test candidate values for `num_iteration` hyperparameter
if MODELING_MODE:
    fig, errors_qe, errors_te = plot_som_convergence_over_epochs(
        SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=False, decay_function=decay_function),
        X_scaled,
        fit_type=SOM_FIT_METHOD, te_ceiling=.1,
        epoch_step_from=2, epoch_step_to=50, epoch_step=1,
        figsize=(16, 5), show_fig=True, verbose=VERBOSE
    )
    print(f"\nQE (first -> last): \t {errors_qe[0]:.2f} -> {errors_qe[-1]:.2f}")
    print(f"TE (first -> last): \t {errors_te[0]:.2f} -> {errors_te[-1]:.2f}")
else: print("Skipping model candidate parameters evaluation.")

Skipping model candidate parameters evaluation.


In [8]:
# set selected `num_iteration` as epoch
epoch = 50

In [9]:
# fit SOM representation
som_rep = SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=VERBOSE, decay_function=decay_function) \
    .fit_online(X_scaled, num_iteration=epoch)

 [ 7500 / 7500 ] 100% - 0:00:00 left 
 quantization error: 0.4326757938872267

 An SOM representation has been fitted as follows:
------------------------------------------------------- 

Fit strategy: online 

Hyperparameters of SOM: 

{'input_len': 4, 'x': 6, 'y': 11, 'sigma': 3.67, 'topology': 'rectangular', 'learning_rate': 0.5, 'decay_function': 'linear_decay_to_zero', 'sigma_decay_function': 'asymptotic_decay', 'neighborhood_function': 'gaussian', 'activation_distance': 'euclidean', 'random_seed': 42, 'num_iteration': 50, 'use_epochs': True, 'random_order': True, 'verbose': True} 

Quality of SOM: 

Quantization Error (QE):	0.4326757938872267
Topographic Error (TE): 	0.0


## Inspection of the learned 2D topology

In [10]:
from utils.plotting import PlotlyHelperArgs

In [11]:
# create Basin
from lilypond import Basin
basin = Basin.from_som_representation(som_rep, random_seed=RANDOM_SEED, verbose=VERBOSE)

In [12]:
# export basin
from utils.export import BasinWithTrainingData
BasinWithTrainingData(dataset_name=DATASET_NAME, basin=basin, X_train=X_scaled).export("_exports/other")

In [12]:
# traditional visuals
plot_args = dict(
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    title="",
)
figTrad1 = basin.legacy_pond().visualize_distance_map(**plot_args, **PlotlyHelperArgs.Figsize(w=925, h=350));
figTrad2 = basin.legacy_pond().visualize_activation_map(**plot_args, **PlotlyHelperArgs.Figsize(w=875, h=350));

In [13]:
EXPORT_DIR = "_exports"
figTrad1.write_image(EXPORT_DIR + "/02_01_lilypond_trad_01.png")
figTrad2.write_image(EXPORT_DIR + "/02_01_lilypond_trad_02.png")

In [14]:
# lilypond visual
basin.pond() \
    .rhizome_layer() \
    .pad_layer() \
    .petal_layer() \
    .visualize(width=800, height=400);

## Detailed inspection

In [15]:
plot_args = dict(
    **PlotlyHelperArgs.Figsize(w=400, h=250),
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    showlegend=False
)

In [16]:
fig1 = basin.pond() \
    .pad_layer() \
    .visualize(**plot_args);

In [17]:
fig2 = basin.pond() \
    .pad_layer(gap="nogap") \
    .visualize(**plot_args);

In [18]:
fig3 = basin.pond() \
    .pad_layer(gap="nogap") \
    .petal_layer() \
    .visualize(**plot_args);

In [19]:
fig4 = basin.pond() \
    .pad_layer(gap="nogap") \
    .rhizome_layer() \
    .visualize(**plot_args);

In [20]:
fig5 = basin.pond() \
    .pad_layer(gap="nogap") \
    .rhizome_layer(violations_only=True) \
    .visualize(**plot_args);

In [21]:
EXPORT_DIR = "_exports"
fig1.write_image(EXPORT_DIR + "/02_01_lilypond_01.png")
fig2.write_image(EXPORT_DIR + "/02_01_lilypond_02.png")
fig3.write_image(EXPORT_DIR + "/02_01_lilypond_03.png")
fig4.write_image(EXPORT_DIR + "/02_01_lilypond_04.png")
fig5.write_image(EXPORT_DIR + "/02_01_lilypond_05.png")

---

### The below cells are not part of the experiment. They are used to register the model in Databricks and Bianor for further interactive investigation.

---

In [ ]:
# TODO: The above model is not yet registered -> register as a new version.

## Register representation model in Databricks

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
CATALOG = "workspace"
SCHEMA = "lilypond_experiments"
MODEL_NAME = "som-iris"
MODEL_PATH = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
MODEL_PATH

In [ ]:
import mlflow
import pandas as pd
from typing import Any

EXPERIMENT_NAME = "/Users/matebalogh@ophelia-rnd.dev/Bianor_LilypondExperiments_Iris"
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

class MLflowSomModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model:SomRepresentation, scaler):
        self.model = model
        self.scaler = scaler
    def predict(self, context, model_input, params: dict[str, Any] | None = None):
        """First transforms the input data via scaler, then predicts the winner node of the SOM."""
        return [self.model.som.winner(x) for x in self.scaler.transform(model_input.to_numpy())]

if MODEL_REGISTRATION_MODE:
    with mlflow.start_run():
        model = MLflowSomModelWrapper(som_rep, scaler)

        mlflow.log_metric("QE", som_rep.quantization_error)
        mlflow.log_metric("TE", som_rep.topographic_error)

        mlflow.pyfunc.log_model(
            python_model=model,
            name=MODEL_NAME,
            input_example=pd.DataFrame(X_scaled[:3]),
            pip_requirements=[
                "numpy",
                "pandas",
                "scikit-learn==1.5.2",
                "mlflow",
                "minisom",
            ],
            registered_model_name=MODEL_PATH
        )
else: print("Skipping MLflow model registration.")

In [ ]:
version = 1
registered_model = f"{MODEL_NAME}/{version}"
registered_model

## Register metadata in Bianor

In [ ]:
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.getOrCreate()

In [ ]:
from bianor_databricks_kit import BianorRecordManager
bianor_recorder = BianorRecordManager(catalog=CATALOG, schema=SCHEMA, spark=spark)

In [ ]:
registered_model_location = f"{CATALOG}.{SCHEMA}.{registered_model}"
registered_model_location

In [ ]:
if MODEL_REGISTRATION_MODE:
    bianor_recorder.new_representation(name=f"{DATASET_NAME} Representation", som_model_location=registered_model_location, features_location="c.s.t") # FIXME
else: print("Skipping Bianor metadata registration.")